In [ ]:
import uproot
import ROOT
import numpy as np
import matplotlib.pyplot as plt
import awkward as ak

In [ ]:
POT_MC = 2.43007e+20
POT_OFFBEAM = 2.11e+20
POT_WIREMOD = 1.03135e+21
#1.03135e+21

In [ ]:
NBINS = 100
ANGLE_CUT = 90
HIGH_X = 60

In [ ]:
NBINS_DEPE = 50
HIGH_DEPE = 200

In [ ]:
particle = "_protons"
#particle = "_mu"
cut = "_CUT"

In [ ]:
#mode_vars = ['STANDARD']
#mode_vars = ['GAIN','BETA','ALPHA','R', 'STANDARD', f'WIREMOD{cut}']
#var_color = ['blue','deepskyblue','darkgreen','lime','red','orange','purple','pink','gray', 'gold']

#mode_vars = ['GAIN','BETA', 'BETA','ALPHA','R','STANDARD']
#var_color = ['blue','deepskyblue','darkgreen','lime','red','orange','purple','pink','gray']

mode_vars = ['STANDARD', 'SCALED']
var_color = ['cornflowerblue', 'orange']
run = 2

In [ ]:
RR_STEP = 1
LENGTH_COND = 5.

dedx_template_file = ROOT.TFile.Open("RefCurvesChi2.root","READ")
if particle == '_protons' : dedx_range = dedx_template_file.Get("dedx_range_pro")
if particle == '_mu' : dedx_range = dedx_template_file.Get("dedx_range_mu")

#nbins = [90, 120, 120, 100, 80, 80, 80, 60, 60, 60, 60, 60, 60, 60, 60, 60, 60, 60, 60, 30, 30, 30, 30, 30, 30]
#xhighs = [30, 30, 30, 25, 20, 20, 20, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 10, 10, 10, 10, 10, 10]

#nbins = 60
#xhighs = 15

#for RR in range(0,1) :
for RR in np.arange(2, 5, RR_STEP):

    low_dedx_t = dedx_range.GetBinContent(dedx_range.FindBin(RR))
    high_dedx_t = dedx_range.GetBinContent(dedx_range.FindBin(RR+RR_STEP)) 

    #NBINS = nbins[RR-1]
    #HIGH_X = xhighs[RR-1]

    #NBINS = nbins
    #HIGH_X = xhighs

    NBINS = 90
    HIGH_X = 60

    fig = plt.figure()

    gs = fig.add_gridspec(2, 1, height_ratios=[8,2], hspace=0)

    print('RR interval: [',RR,RR+RR_STEP,')')

    filename = f'ROOT_TREES_DEDX/RUN2/varMC_STANDARD_RUN{run}_DATA{cut}.root'

    print(filename)

    file = uproot.open(filename)
    tree = file['tree']
    print("num entries:", tree.num_entries)
    print("keys:", tree.keys())
    arrays = tree.arrays(
            ["slice/_protons/_protons._dedx", 
            "slice/_protons/_protons._rr",
            "slice/_mu/_mu._dedx", 
            "slice/_mu/_mu._rr",
            "slice/_protons/_protons._theta_xw",
            "slice/_mu/_mu._theta_xw",
            "slice/_protons/_protons._pitch",
            "slice/_protons/_protons._mult",
            "slice/_protons/_protons._dqdx",
            "slice/_protons/_protons._phi",
            "slice/_mu/_mu._pitch",
            "slice/_mu/_mu._mult",
            "slice/_mu/_mu._dqdx",
            "slice/_mu/_mu._phi",
            "slice/_protons/_protons._length"],
            library="ak"
            )

    #theta_broadcast, _ = ak.broadcast_arrays(arrays[f"slice/{particle}/{particle}._theta_xw"], arrays[f"slice/{particle}/{particle}._dedx"])

    #theta_flat = ak.to_numpy(ak.flatten(theta_broadcast, axis=None))
    rr_flat   = ak.to_numpy(ak.flatten(arrays[f"slice/{particle}/{particle}._rr"], axis=None))
    dedx_flat = ak.to_numpy(ak.flatten(arrays[f"slice/{particle}/{particle}._dedx"], axis=None))

    length_broadcast,_ = ak.broadcast_arrays(arrays[f"slice/{particle}/{particle}._length"], arrays[f"slice/{particle}/{particle}._rr"])
    length_flat = ak.to_numpy(ak.flatten(length_broadcast, axis=None))

    #mask = (rr_flat >= RR) & (rr_flat < RR + RR_STEP) & (np.abs(theta_flat)*180/np.pi < ANGLE_CUT)
    mask = (rr_flat >= RR) & (rr_flat < RR + RR_STEP) & (length_flat > LENGTH_COND)
    dedx_thisRR = dedx_flat[mask]

    print('DATA: ',len(dedx_thisRR),'hits')
    
    h_data = ROOT.TH1D(f"h_data_rr{RR}","",NBINS,0,HIGH_X)
    h_data.Sumw2()
    data_np = np.array(dedx_thisRR)
    #h_data.FillN(data_np.size,data_np,np.ones_like(data_np))
    for hit_data in data_np : h_data.Fill(hit_data)

    h_data.Scale(1. / h_data.Integral("width"))

    bin_centers_DATA = []
    counts_DATA = []
    errors_DATA = []

    for bin in range(1,NBINS + 1):
        bin_centers_DATA.append(h_data.GetBinCenter(bin))
        counts_DATA.append(h_data.GetBinContent(bin))
        errors_DATA.append(h_data.GetBinError(bin))

    filename = f'ROOT_TREES_DEDX/RUN2/varMC_STANDARD_RUN{run}_OFFBEAM{cut}.root'

    print(filename)

    file = uproot.open(filename)
    tree = file['tree']
    arrays = tree.arrays(
            ["slice/_protons/_protons._dedx", 
            "slice/_protons/_protons._rr",
            "slice/_mu/_mu._dedx", 
            "slice/_mu/_mu._rr",
            "slice/_protons/_protons._theta_xw",
            "slice/_mu/_mu._theta_xw",
            "slice/_protons/_protons._pitch",
            "slice/_protons/_protons._mult",
            "slice/_protons/_protons._dqdx",
            "slice/_protons/_protons._phi",
            "slice/_mu/_mu._pitch",
            "slice/_mu/_mu._mult",
            "slice/_mu/_mu._dqdx",
            "slice/_mu/_mu._phi",
            "slice/_protons/_protons._length"],
            library="ak"
            )
    
    #theta_broadcast, _ = ak.broadcast_arrays(arrays[f"slice/{particle}/{particle}._theta_xw"], arrays[f"slice/{particle}/{particle}._dedx"])

    rr_flat   = ak.to_numpy(ak.flatten(arrays[f"slice/{particle}/{particle}._rr"],   axis=None))
    dedx_flat = ak.to_numpy(ak.flatten(arrays[f"slice/{particle}/{particle}._dedx"], axis=None))

    length_broadcast,_ = ak.broadcast_arrays(arrays[f"slice/{particle}/{particle}._length"], arrays[f"slice/{particle}/{particle}._rr"])
    length_flat = ak.to_numpy(ak.flatten(length_broadcast, axis=None))
    
    #theta_flat = ak.to_numpy(ak.flatten(theta_broadcast, axis=None))
    #mask = (rr_flat >= RR) & (rr_flat < RR+RR_STEP) & (np.abs(theta_flat)*180/np.pi < ANGLE_CUT)

    mask = (rr_flat >= RR) & (rr_flat < RR+RR_STEP) & (length_flat > LENGTH_COND)
    dedx_thisRR_offbeam = dedx_flat[mask]
    weights_OFFBEAM_dedx = np.full(len(dedx_thisRR_offbeam), 1. / POT_OFFBEAM)

    print('OFFBEAM: ',len(dedx_thisRR_offbeam),'hits')

    plt.subplot(gs[0])
    plt.xlim(0,HIGH_X)
    plt.errorbar(bin_centers_DATA,counts_DATA,yerr=errors_DATA,fmt='o', ms=2., capsize=1., capthick=1., elinewidth=1., color = 'black', label = 'DATA')
    plt.axvline(low_dedx_t, color='green', linestyle='--',label=f'dE/dx reference {RR} cm')
    plt.axvline(high_dedx_t, color ='violet',linestyle='--', label=f'dE/dx reference {RR+RR_STEP} cm')

    m = -1
    for mode in mode_vars:

        for n,sigma in enumerate(['_plus1sigma','_minus1sigma']):

            if (n == 0 and mode == 'STANDARD') or (n == 0 and mode == f'WIREMOD{cut}') or (n == 0 and mode == 'SCALED') or (n == 0 and mode == 'SCALED_OLD_SPLINE') : continue
            if (n == 1 and mode == 'STANDARD') or (n == 1 and mode == f'WIREMOD{cut}') or (n == 1 and mode == 'SCALED') or (n == 1 and mode == 'SCALED_OLD_SPLINE') : sigma = ''

            m = m+1

            filename = f'ROOT_TREES_DEDX/RUN2/varMC_{mode}_RUN{run}{cut}{sigma}.root'

            print(filename)

            file = uproot.open(filename)

            tree = file['tree']
            #print(tree.keys())

            arrays = tree.arrays(
            ["slice/_protons/_protons._dedx", 
            "slice/_protons/_protons._rr",
            "slice/_mu/_mu._dedx", 
            "slice/_mu/_mu._rr",
            "slice/_protons/_protons._theta_xw",
            "slice/_mu/_mu._theta_xw",
            "slice/_protons/_protons._pitch",
            "slice/_protons/_protons._mult",
            "slice/_protons/_protons._dqdx",
            "slice/_protons/_protons._phi",
            "slice/_mu/_mu._pitch",
            "slice/_mu/_mu._mult",
            "slice/_mu/_mu._dqdx",
            "slice/_mu/_mu._phi",
            "slice/_protons/_protons._pdg",
            "slice/_protons/_protons._mediana",
            "slice/_mu/_mu._pdg",
            "slice/_mu/_mu._mediana",
            "slice/_protons/_protons._length"],
            library="ak"
            )

            #theta_broadcast, _ = ak.broadcast_arrays(arrays[f"slice/{particle}/{particle}._theta_xw"], arrays[f"slice/{particle}/{particle}._dedx"])

            rr_flat   = ak.to_numpy(ak.flatten(arrays[f"slice/{particle}/{particle}._rr"], axis=None))
            dedx_flat = ak.to_numpy(ak.flatten(arrays[f"slice/{particle}/{particle}._dedx"], axis=None))

            length_broadcast,_ = ak.broadcast_arrays(arrays[f"slice/{particle}/{particle}._length"], arrays[f"slice/{particle}/{particle}._rr"])
            length_flat = ak.to_numpy(ak.flatten(length_broadcast, axis=None))

            #theta_flat = ak.to_numpy(ak.flatten(theta_broadcast, axis=None))
            #mask = (rr_flat >= RR) & (rr_flat < RR + RR_STEP) & (np.abs(theta_flat)*180/np.pi < ANGLE_CUT)

            mask = (rr_flat >= RR) & (rr_flat < RR + RR_STEP) & (length_flat > LENGTH_COND)
            dedx_thisRR = dedx_flat[mask]
            POT = 0
            if mode == f"WIREMOD{cut}" : POT = POT_WIREMOD 
            else : POT = POT_MC
            weights_MC_dedx = np.full(len(dedx_thisRR), 1. / POT)

            print(f'MC {mode}: ',len(dedx_thisRR),'hits')

            mc_off = np.concatenate([dedx_thisRR, dedx_thisRR_offbeam])
            weights_mc_off = np.concatenate([weights_MC_dedx, weights_OFFBEAM_dedx])
            n_mc_off, bins = np.histogram(
            mc_off,
            bins=NBINS,
            range=(0, HIGH_X),
            weights=weights_mc_off,
            density=True
            )

            print(f'MC {mode}: [',n_mc_off,']')

            bin_centers = 0.5 * (bins[:-1] + bins[1:])
            bin_widths = np.diff(bins)

            ratio = np.zeros_like(counts_DATA)
            ratio_err = np.zeros_like(counts_DATA)

            for k in range(len(counts_DATA)):
                if n_mc_off[k] > 0:
                    ratio[k] = counts_DATA[k] / n_mc_off[k]

            sigma_n = 0
            if sigma == '_plus1sigma' : sigma_n = 1
            elif sigma == '_minus1sigma' : sigma_n = -1

            plt.subplot(gs[0])

            lls = "-"
            llw = 2
            #if mode == 'STANDARD' : 
            #    lls = '--'
            #    llw = 3
            
            plt.hist(mc_off, bins=NBINS, range=(0, HIGH_X), weights=weights_mc_off, density=True, histtype='step', lw=llw, ls=lls, label=fr'MC var {mode} ${sigma_n}\sigma$', color = var_color[m], alpha = 0.8)

            if mode == 'SCALED':

                pdg_broadcast,_ = ak.broadcast_arrays(arrays[f"slice/{particle}/{particle}._pdg"], arrays[f"slice/{particle}/{particle}._rr"])
    
                pdg_flat = ak.to_numpy(ak.flatten(pdg_broadcast, axis=None))
                print(pdg_flat)

                mask_pdg13 = (rr_flat >= RR) & (rr_flat < RR + RR_STEP) & (np.abs(pdg_flat) == 13.) & (length_flat > LENGTH_COND)
                dedx_thisRR_pdg13 = dedx_flat[(mask_pdg13)]
                weights_pdg13 = np.full(len(dedx_thisRR_pdg13), 1. / POT)
                print('n hit pdg == 13', len(dedx_thisRR_pdg13))

                mask_pdg211 = (rr_flat >= RR) & (rr_flat < RR + RR_STEP) & (np.abs(pdg_flat) == 211.) & (length_flat > LENGTH_COND)
                dedx_thisRR_pdg211 = dedx_flat[(mask_pdg211)]
                weights_pdg211 = np.full(len(dedx_thisRR_pdg211), 1. / POT)
                print('n hit pdg == 211', len(dedx_thisRR_pdg211))

                mask_pdg11 = (rr_flat >= RR) & (rr_flat < RR + RR_STEP) & (np.abs(pdg_flat) == 11.) & (length_flat > LENGTH_COND)
                dedx_thisRR_pdg11 = dedx_flat[(mask_pdg11)]
                weights_pdg11 = np.full(len(dedx_thisRR_pdg11), 1. / POT)
                print('n hit pdg == 11', len(dedx_thisRR_pdg11))

                mask_pdg2212 = (rr_flat >= RR) & (rr_flat < RR + RR_STEP) & (np.abs(pdg_flat) == 2212.) & (length_flat > LENGTH_COND)
                dedx_thisRR_pdg2212 = dedx_flat[(mask_pdg2212)]
                weights_pdg2212 = np.full(len(dedx_thisRR_pdg2212), 1. / POT)
                print('n hit pdg == 2212', len(dedx_thisRR_pdg2212))

                mask_pdg_other = (
                    (rr_flat >= RR) & (rr_flat < RR + RR_STEP)
                    & (np.abs(pdg_flat) != 13.)
                    & (np.abs(pdg_flat) != 211.)
                    & (np.abs(pdg_flat) != 11.)
                    & (np.abs(pdg_flat) != 2212.)
                )
                dedx_thisRR_pdgother = dedx_flat[mask_pdg_other]
                weights_pdgother = np.full(len(dedx_thisRR_pdgother), 1. / POT)

                plt.hist(
                    [dedx_thisRR_pdg13,dedx_thisRR_pdg211,dedx_thisRR_pdg11,dedx_thisRR_pdg2212,dedx_thisRR_pdgother,dedx_thisRR_offbeam],
                    bins=NBINS,
                    range=(0, HIGH_X),
                    weights = [weights_pdg13,weights_pdg211,weights_pdg11,weights_pdg2212,weights_pdgother,weights_OFFBEAM_dedx],
                    label=[r'$\mu$', r'$\pi$', r'$e$', r'$p$', 'other', 'offbeam'],
                    color=['violet','red','green','blue', 'limegreen','gray'],
                    stacked = True,
                    histtype= 'bar',
                    density = True,
                    alpha = 0.7
                )

            if mode == 'SCALED' and False:

                mediana_broadcast,_ = ak.broadcast_arrays(arrays[f"slice/{particle}/{particle}._mediana"], arrays[f"slice/{particle}/{particle}._rr"])
                mediana_flat = ak.to_numpy(ak.flatten(mediana_broadcast, axis=None))

                mask_interacting = (rr_flat >= RR) & (rr_flat < RR + RR_STEP) & (mediana_flat <= 3.4) & (length_flat > LENGTH_COND) #8.3 #3.4
                mask_stopping = (rr_flat >= RR) & (rr_flat < RR + RR_STEP) & (mediana_flat > 3.4) & (length_flat > LENGTH_COND)

                dedx_interacting = dedx_flat[(mask_interacting)]
                weights_interacting = np.full(len(dedx_interacting), 1./POT)

                dedx_stopping = dedx_flat[(mask_stopping)]
                weights_stopping = np.full(len(dedx_stopping), 1./POT)

                plt.hist(
                    [dedx_interacting,dedx_stopping,dedx_thisRR_offbeam],
                    bins=NBINS,
                    range=(0, HIGH_X),
                    weights = [weights_interacting,weights_stopping,weights_OFFBEAM_dedx],
                    label=[r'interacting', r'stopping', 'offbeam'],
                    color=['green','deepskyblue','gray'],
                    stacked = True,
                    histtype= 'bar',
                    density = True,
                    alpha = 0.7
                )

            
            plt.setp(plt.gca().get_xticklabels(), visible=False)
            plt.gca().tick_params(labelbottom=False)
            ax = plt.gca()
            yticks = ax.get_yticks()
            # keep only ticks strictly greater than the axis minimum, so the bottom one is dropped
            ax.set_yticks(yticks[yticks > ax.get_ylim()[0]])
            plt.ylabel('entries (area norm.)', fontsize=14)
            plt.yticks(fontsize=14)

            if particle == '_mu' : par = 'MUONS'
            elif particle == '_protons' : par = 'PROTONS'
            plt.title('RUN {} - {} - dE/dx for RR bin [{},{}) cm'.format(run,par,RR,RR+RR_STEP), fontsize=18)

            plt.subplot(gs[1])

            plt.xlim(0,HIGH_X)

            dx = bin_centers[1] - bin_centers[0]

            edges = np.concatenate([
            [bin_centers[0] - dx/2],
            bin_centers[:-1] + dx/2,
            [bin_centers[-1] + dx/2]
            ])

            plt.stairs(ratio,edges,fill=False,lw=2,color=var_color[m])

            plt.fill_between(
            edges[:-1],
            1,
            ratio,
            step='post',
            alpha=0.4,
            color=var_color[m],
            lw=0
            )

            plt.axhline(1.0, color='black', linestyle='--')
            #plt.axvline(low_dedx_t, color='green', linestyle='--')
            #plt.axvline(high_dedx_t, color ='violet',linestyle='--')

            plt.xlabel('dE/dx [MeV/cm]', fontsize=14)
            plt.ylabel(r'$\frac{DATA}{MC}$', fontsize=14)
            plt.xticks(fontsize=14, rotation=60)
            plt.yticks(fontsize=14)
            plt.ylim(0,2)


    plt.subplot(gs[0])  
    plt.legend(loc='upper right',fontsize = 8, ncols = 1)
    #plt.savefig(f'DEDX_VAR/rr{RR}{particle}_STANDARD_RUN{run}_RESCALED_CUT.pdf',format='pdf',bbox_inches='tight')



        


In [ ]:
NBINS_CHI2 = 50
XHIGH_CHI2 = 100

add_string = 'CHI2EXPLORATION'

chi2as = 'as_pro_05'

fig = plt.figure()

gs = fig.add_gridspec(2, 1, height_ratios=[8,2], hspace=0)

filename = f'ROOT_TREES_DEDX/RUN2/varMC_STANDARD_RUN{run}_DATA{cut}_{add_string}.root'
file = uproot.open(filename)
tree = file['tree']
print("num entries:", tree.num_entries)
print("keys:", tree.keys())
arrays = tree.arrays(
        ["slice/_protons/_protons._dedx", 
        "slice/_protons/_protons._rr",
        "slice/_mu/_mu._dedx", 
        "slice/_mu/_mu._rr",
        "slice/_protons/_protons._theta_xw",
        "slice/_mu/_mu._theta_xw",
        "slice/_protons/_protons._depE",
        "slice/_mu/_mu._depE",
        "slice/_protons/_protons._chi2_as_mu",
        "slice/_mu/_mu._chi2_as_mu",
        "slice/_protons/_protons._chi2_as_pro",
        "slice/_mu/_mu._chi2_as_pro",
        "slice/_protons/_protons._chi2_as_mu_05",
        "slice/_mu/_mu._chi2_as_mu_05",
        "slice/_protons/_protons._chi2_as_pro_05",
        "slice/_mu/_mu._chi2_as_pro_05"],
        library="ak"
        )

chi2 = ak.to_numpy(ak.flatten(arrays[f"slice/{particle}/{particle}._chi2_{chi2as}"], axis=None))
    
h_data = ROOT.TH1D("h_data_chi2","",NBINS_CHI2,0,XHIGH_CHI2)
h_data.Sumw2()
data_np = np.array(chi2)
for hit_data in data_np : h_data.Fill(hit_data)

h_data.Scale(1. / h_data.Integral("width"))

bin_centers_DATA = []
counts_DATA = []
errors_DATA = []

for bin in range(1,NBINS_CHI2 + 1):
    bin_centers_DATA.append(h_data.GetBinCenter(bin))
    counts_DATA.append(h_data.GetBinContent(bin))
    errors_DATA.append(h_data.GetBinError(bin))


filename = f'ROOT_TREES_DEDX/RUN2/varMC_STANDARD_RUN{run}_OFFBEAM{cut}_{add_string}.root'
file = uproot.open(filename)
tree = file['tree']
arrays = tree.arrays(
        ["slice/_protons/_protons._dedx", 
        "slice/_protons/_protons._rr",
        "slice/_mu/_mu._dedx", 
        "slice/_mu/_mu._rr",
        "slice/_protons/_protons._theta_xw",
        "slice/_mu/_mu._theta_xw",
        "slice/_protons/_protons._depE",
        "slice/_mu/_mu._depE",
        "slice/_protons/_protons._chi2_as_mu",
        "slice/_mu/_mu._chi2_as_mu",
        "slice/_protons/_protons._chi2_as_pro",
        "slice/_mu/_mu._chi2_as_pro",
        "slice/_protons/_protons._chi2_as_mu_05",
        "slice/_mu/_mu._chi2_as_mu_05",
        "slice/_protons/_protons._chi2_as_pro_05",
        "slice/_mu/_mu._chi2_as_pro_05"],
        library="ak"
        )

chi2_offbeam = ak.to_numpy(ak.flatten(arrays[f"slice/{particle}/{particle}._chi2_{chi2as}"], axis=None))

weights_OFFBEAM_chi2 = np.full(len(chi2_offbeam), 1. / POT_OFFBEAM)

plt.subplot(gs[0])
plt.errorbar(bin_centers_DATA,counts_DATA,yerr=errors_DATA,fmt='o', ms=2., capsize=1., capthick=1., elinewidth=1., color = 'black', label = 'DATA')

m = -1
for mode in mode_vars:

    for n,sigma in enumerate(['_plus1sigma','_minus1sigma']):

        if (n == 0 and mode == 'STANDARD') or (n == 0 and mode == f'WIREMOD{cut}') or (n == 0 and mode == 'SCALED') or (n == 0 and mode == 'SCALED_OLD_SPLINE') : continue
        if (n == 1 and mode == 'STANDARD') or (n == 1 and mode == f'WIREMOD{cut}') or (n == 1 and mode == 'SCALED') or (n == 1 and mode == 'SCALED_OLD_SPLINE') : sigma = ''

        m = m+1

        filename = f'ROOT_TREES_DEDX/RUN2/varMC_{mode}_RUN{run}{cut}{sigma}_{add_string}.root'

        print(filename)

        file = uproot.open(filename)

        tree = file['tree']
        #print(tree.keys())

        arrays = tree.arrays(
        ["slice/_protons/_protons._dedx", 
        "slice/_protons/_protons._rr",
        "slice/_mu/_mu._dedx", 
        "slice/_mu/_mu._rr",
        "slice/_protons/_protons._theta_xw",
        "slice/_mu/_mu._theta_xw",
        "slice/_protons/_protons._depE",
        "slice/_mu/_mu._depE",
        "slice/_protons/_protons._chi2_as_mu",
        "slice/_mu/_mu._chi2_as_mu",
        "slice/_protons/_protons._chi2_as_pro",
        "slice/_mu/_mu._chi2_as_pro",
        "slice/_protons/_protons._pdg",
        "slice/_protons/_protons._mediana",
        "slice/_mu/_mu._pdg",
        "slice/_mu/_mu._mediana",
        "slice/_protons/_protons._chi2_as_mu_05",
        "slice/_mu/_mu._chi2_as_mu_05",
        "slice/_protons/_protons._chi2_as_pro_05",
        "slice/_mu/_mu._chi2_as_pro_05"],
        library="ak"
        )

        chi2 = ak.to_numpy(ak.flatten(arrays[f"slice/{particle}/{particle}._chi2_{chi2as}"], axis=None))
        pdg = ak.to_numpy(ak.flatten(arrays[f"slice/{particle}/{particle}._pdg"], axis=None))
        mediana = ak.to_numpy(ak.flatten(arrays[f"slice/{particle}/{particle}._mediana"], axis=None))

        POT = 0
        if mode == f"WIREMOD{cut}" : POT = POT_WIREMOD 
        else : POT = POT_MC
        weights_MC_chi2 = np.full(len(chi2), 1. / POT)

        plt.subplot(gs[0])

        if mode == 'SCALED' and True:
            chi2_pdg13 = chi2[(np.abs(pdg) == 13)]
            weights_pdg13 = np.full(len(chi2_pdg13), 1./POT)
            print(len(chi2_pdg13))

            chi2_pdg211 = chi2[(np.abs(pdg) == 211)]
            weights_pdg211 = np.full(len(chi2_pdg211), 1./POT)

            chi2_pdg11 = chi2[(np.abs(pdg) == 11)]
            weights_pdg11 = np.full(len(chi2_pdg11), 1./POT)

            chi2_pdg2212 = chi2[(np.abs(pdg) == 2212)]
            weights_pdg2212 = np.full(len(chi2_pdg2212), 1./POT)

            chi2_pdg_other = chi2[
                (np.abs(pdg) != 13) & 
                (np.abs(pdg) != 211) &
                (np.abs(pdg) != 11) &
                (np.abs(pdg) != 2212) 
                ]
            weights_pdg_other = np.full(len(chi2_pdg_other), 1./POT)

            plt.hist(
                    [chi2_pdg13, chi2_pdg211, chi2_pdg11, chi2_pdg2212, chi2_pdg_other ,chi2_offbeam],
                    bins=NBINS_CHI2,
                    range=(0, XHIGH_CHI2),
                    weights = [weights_pdg13, weights_pdg211, weights_pdg11, weights_pdg2212, weights_pdg_other ,weights_OFFBEAM_chi2],
                    label=[r'$\mu$', r'$\pi$', r'$e$', r'$p$', 'other', 'offbeam'],
                    color=['violet','red','green','blue', 'limegreen','gray'],
                    stacked = True,
                    histtype= 'bar',
                    density = True,
                    alpha = 0.7
                )
            
        if mode == 'SCALED' and False:
            chi2_interacting = chi2[(mediana <= 8.3)]
            weights_interacting = np.full(len(chi2_interacting), 1./POT)

            chi2_stopping = chi2[(mediana > 8.3)]
            weights_stopping = np.full(len(chi2_stopping), 1./POT)


            plt.hist(
                    [chi2_interacting, chi2_stopping,chi2_offbeam],
                    bins=NBINS_CHI2,
                    range=(0, XHIGH_CHI2),
                    weights = [weights_interacting, weights_stopping ,weights_OFFBEAM_chi2],
                    label=['interacting', 'stopping', 'offbeam'],
                    color=['green', 'deepskyblue','gray'],
                    stacked = True,
                    histtype= 'bar',
                    density = True,
                    alpha = 0.7
                )

        mc_off = np.concatenate([chi2, chi2_offbeam])
        weights_mc_off = np.concatenate([weights_MC_chi2, weights_OFFBEAM_chi2])
        n_mc_off, bins = np.histogram(
        mc_off,
        bins=NBINS_CHI2,
        range=(0, XHIGH_CHI2),
        weights=weights_mc_off,
        density=True
        )

        bin_centers = 0.5 * (bins[:-1] + bins[1:])
        bin_widths = np.diff(bins)

        ratio = np.zeros_like(counts_DATA)
        ratio_err = np.zeros_like(counts_DATA)

        for k in range(len(counts_DATA)):
            if n_mc_off[k] > 0:
                ratio[k] = counts_DATA[k] / n_mc_off[k]

        sigma_n = 0
        if sigma == '_plus1sigma' : sigma_n = 1
        elif sigma == '_minus1sigma' : sigma_n = -1


        lls = "-"
        llw = 2
        #if mode == 'STANDARD' : 
        #    lls = '--'
        #    llw = 3
            
        plt.hist(mc_off, bins=NBINS_CHI2, range=(0, XHIGH_CHI2), weights=weights_mc_off, density=True, histtype='step', lw=llw, ls=lls, label=fr'MC var {mode} ${sigma_n}\sigma$', color = var_color[m], alpha = 0.8)
        plt.setp(plt.gca().get_xticklabels(), visible=False)
        plt.gca().tick_params(labelbottom=False)
        ax = plt.gca()
        yticks = ax.get_yticks()
        # keep only ticks strictly greater than the axis minimum, so the bottom one is dropped
        ax.set_yticks(yticks[yticks > ax.get_ylim()[0]])
        plt.ylabel('entries (area norm.)', fontsize=14)
        plt.yticks(fontsize=14)

        plt.subplot(gs[1])

        dx = bin_centers[1] - bin_centers[0]

        edges = np.concatenate([
        [bin_centers[0] - dx/2],
        bin_centers[:-1] + dx/2,
        [bin_centers[-1] + dx/2]
        ])

        plt.stairs(ratio,edges,fill=False,lw=2,color=var_color[m])

        plt.fill_between(
        edges[:-1],
        1,
        ratio,
        step='post',
        alpha=0.4,
        color=var_color[m],
        lw=0
        )

        plt.axhline(1.0, color='black', linestyle='--')

        if chi2as == 'as_mu' : plt.xlabel(r'$\chi^2_\mu$', fontsize=14)
        if chi2as == 'as_pro' : plt.xlabel(r'$\chi^2_p$', fontsize=14)
        plt.ylabel(r'$\frac{DATA}{MC}$', fontsize=14)
        plt.xticks(fontsize=14, rotation=60)
        plt.yticks(fontsize=14)
        plt.ylim(0,2)


plt.subplot(gs[0])  
plt.legend(loc='upper right',fontsize = 6, ncols = 1)
plt.savefig(f'DEDX_VAR/chi2{particle}_{chi2as}{cut}_CHi2EXPLORATION.pdf',format='pdf',bbox_inches='tight')



        


In [ ]:

fig = plt.figure()

gs = fig.add_gridspec(2, 1, height_ratios=[8,2], hspace=0)

filename = f'ROOT_TREES_DEDX/RUN2/varMC_STANDARD_RUN{run}_DATA{cut}.root'
file = uproot.open(filename)
tree = file['tree']
print("num entries:", tree.num_entries)
print("keys:", tree.keys())
arrays = tree.arrays(
        ["slice/_protons/_protons._dedx", 
        "slice/_protons/_protons._rr",
        "slice/_mu/_mu._dedx", 
        "slice/_mu/_mu._rr",
        "slice/_protons/_protons._theta_xw",
        "slice/_mu/_mu._theta_xw",
        "slice/_protons/_protons._depE",
        "slice/_mu/_mu._depE"],
        library="ak"
        )

depE = ak.to_numpy(ak.flatten(arrays[f"slice/{particle}/{particle}._depE"], axis=None))
    
h_data = ROOT.TH1D("h_data_depE","",NBINS_DEPE,0,HIGH_DEPE)
h_data.Sumw2()
data_np = np.array(depE)
for hit_data in data_np : h_data.Fill(hit_data)

h_data.Scale(1. / h_data.Integral("width"))

bin_centers_DATA = []
counts_DATA = []
errors_DATA = []

for bin in range(1,NBINS_DEPE + 1):
    bin_centers_DATA.append(h_data.GetBinCenter(bin))
    counts_DATA.append(h_data.GetBinContent(bin))
    errors_DATA.append(h_data.GetBinError(bin))


filename = f'ROOT_TREES_DEDX/RUN2/varMC_STANDARD_RUN{run}_OFFBEAM{cut}.root'
file = uproot.open(filename)
tree = file['tree']
arrays = tree.arrays(
        ["slice/_protons/_protons._dedx", 
        "slice/_protons/_protons._rr",
        "slice/_mu/_mu._dedx", 
        "slice/_mu/_mu._rr",
        "slice/_protons/_protons._theta_xw",
        "slice/_mu/_mu._theta_xw",
        "slice/_protons/_protons._depE",
        "slice/_mu/_mu._depE"],
        library="ak"
        )

depE_offbeam = ak.to_numpy(ak.flatten(arrays[f"slice/{particle}/{particle}._depE"], axis=None))

weights_OFFBEAM_depE = np.full(len(depE_offbeam), 1. / POT_OFFBEAM)

plt.subplot(gs[0])
plt.errorbar(bin_centers_DATA,counts_DATA,yerr=errors_DATA,fmt='o', ms=2., capsize=1., capthick=1., elinewidth=1., color = 'black', label = 'DATA')

m = -1
for mode in mode_vars:

    for n,sigma in enumerate(['_plus1sigma','_minus1sigma']):

        if (n == 0 and mode == 'STANDARD') or (n == 0 and mode == f'WIREMOD{cut}') or (n == 0 and mode == 'SCALED') : continue
        if (n == 1 and mode == 'STANDARD') or (n == 1 and mode == f'WIREMOD{cut}') or (n == 1 and mode == 'SCALED'): sigma = ''

        m = m+1

        filename = f'ROOT_TREES_DEDX/RUN2/varMC_{mode}_RUN{run}{cut}{sigma}.root'

        print(filename)

        file = uproot.open(filename)

        tree = file['tree']
        #print(tree.keys())

        arrays = tree.arrays(
        ["slice/_protons/_protons._dedx", 
        "slice/_protons/_protons._rr",
        "slice/_mu/_mu._dedx", 
        "slice/_mu/_mu._rr",
        "slice/_protons/_protons._theta_xw",
        "slice/_mu/_mu._theta_xw",
        "slice/_protons/_protons._depE",
        "slice/_mu/_mu._depE"],
        library="ak"
        )

        depE = ak.to_numpy(ak.flatten(arrays[f"slice/{particle}/{particle}._depE"], axis=None))

        POT = 0
        if mode == f"WIREMOD{cut}" : POT = POT_WIREMOD 
        else : POT = POT_MC
        weights_MC_depE = np.full(len(depE), 1. / POT)

        mc_off = np.concatenate([depE, depE_offbeam])
        weights_mc_off = np.concatenate([weights_MC_depE, weights_OFFBEAM_depE])
        n_mc_off, bins = np.histogram(
        mc_off,
        bins=NBINS_DEPE,
        range=(0, HIGH_DEPE),
        weights=weights_mc_off,
        density=True
        )

        bin_centers = 0.5 * (bins[:-1] + bins[1:])
        bin_widths = np.diff(bins)

        ratio = np.zeros_like(counts_DATA)
        ratio_err = np.zeros_like(counts_DATA)

        for k in range(len(counts_DATA)):
            if n_mc_off[k] > 0:
                ratio[k] = counts_DATA[k] / n_mc_off[k]

        sigma_n = 0
        if sigma == '_plus1sigma' : sigma_n = 1
        elif sigma == '_minus1sigma' : sigma_n = -1

        plt.subplot(gs[0])

        lls = "-"
        llw = 2
        #if mode == 'STANDARD' : 
        #    lls = '--'
        #    llw = 3
            
        plt.hist(mc_off, bins=NBINS_DEPE, range=(0, HIGH_DEPE), weights=weights_mc_off, density=True, histtype='step', lw=llw, ls=lls, label=fr'MC var {mode} ${sigma_n}\sigma$', color = var_color[m], alpha = 0.8)
        plt.setp(plt.gca().get_xticklabels(), visible=False)
        plt.gca().tick_params(labelbottom=False)
        ax = plt.gca()
        yticks = ax.get_yticks()
        # keep only ticks strictly greater than the axis minimum, so the bottom one is dropped
        ax.set_yticks(yticks[yticks > ax.get_ylim()[0]])
        plt.ylabel('entries (area norm.)', fontsize=14)
        plt.yticks(fontsize=14)

        plt.subplot(gs[1])

        dx = bin_centers[1] - bin_centers[0]

        edges = np.concatenate([
        [bin_centers[0] - dx/2],
        bin_centers[:-1] + dx/2,
        [bin_centers[-1] + dx/2]
        ])

        plt.stairs(ratio,edges,fill=False,lw=2,color=var_color[m])

        plt.fill_between(
        edges[:-1],
        1,
        ratio,
        step='post',
        alpha=0.4,
        color=var_color[m],
        lw=0
        )

        plt.axhline(1.0, color='black', linestyle='--')

        plt.xlabel('depE [MeV]', fontsize=14)
        plt.ylabel(r'$\frac{DATA}{MC}$', fontsize=14)
        plt.xticks(fontsize=14, rotation=60)
        plt.yticks(fontsize=14)
        plt.ylim(0,2)


plt.subplot(gs[0])  
plt.legend(loc='upper right',fontsize = 8, ncols = 1)
plt.savefig(f'DEDX_VAR/depE{particle}_SCALED.pdf',format='pdf',bbox_inches='tight')



        


In [ ]:
ref_file = ROOT.TFile.Open("RefCurvesChi2.root","READ")
dedx_range_mu = ref_file.Get("dedx_range_mu")
dedx_range_pro = ref_file.Get("dedx_range_pro")
dedx_range_pi = ref_file.Get("dedx_range_pi")
dedx_range_ka = ref_file.Get("dedx_range_ka")


In [ ]:
import math

# Si assume che dedx_range_pro, dedx_range_ka, dedx_range_pi, dedx_range_mu
# siano istogrammi TH1 (PyROOT) gia' definiti/caricati altrove, come nel codice C++ originale.


def chi2_ALG(dEdx, RR, rr_min, rr_max):
    """
    Traduzione Python della funzione C++ chi2_ALG.

    Parametri
    ---------
    dEdx : list[float]
        Valori di dE/dx lungo la traccia.
    RR : list[float]
        Residual range corrispondente ad ogni punto di dEdx.
    rr_min, rr_max : float
        Limiti del residual range da considerare.

    Ritorna
    -------
    list[float]
        [chi2mu, chi2pro, chi2ka, chi2pi] normalizzati per numero di punti usati.
    """

    threshold = 0.5
    max_rr = rr_max
    min_rr = rr_min

    trkdedx = []
    trkres = []
    vpida = []

    n = len(dEdx)
    for i in range(n):
        if i == 0 or i == n - 1:
            continue
        if RR[i] < max_rr and RR[i] > min_rr:
            trkdedx.append(dEdx[i])
            trkres.append(RR[i])

    npt = 0
    chi2pro = 0.0
    chi2ka = 0.0
    chi2pi = 0.0
    chi2mu = 0.0
    avgdedx = 0.0
    PIDA = 0.0
    used_trkres = 0

    for i in range(len(trkdedx)):
        avgdedx += trkdedx[i]

        if trkres[i] < 26:
            PIDA += trkdedx[i] * (trkres[i] ** 0.42)
            vpida.append(trkdedx[i] * (trkres[i] ** 0.42))
            used_trkres += 1

        if trkdedx[i] > 100 or trkdedx[i] < threshold:
            # protegge da grandi pulse height
            continue

        bin_ = dedx_range_pro.FindBin(trkres[i])
        if 1 <= bin_ <= dedx_range_pro.GetNbinsX():
            bincpro = dedx_range_pro.GetBinContent(bin_)
            if bincpro < 1e-6:
                bincpro = (dedx_range_pro.GetBinContent(bin_ - 1) +
                           dedx_range_pro.GetBinContent(bin_ + 1)) / 2

            bincka = dedx_range_ka.GetBinContent(bin_)
            if bincka < 1e-6:
                bincka = (dedx_range_ka.GetBinContent(bin_ - 1) +
                          dedx_range_ka.GetBinContent(bin_ + 1)) / 2

            bincpi = dedx_range_pi.GetBinContent(bin_)
            if bincpi < 1e-6:
                bincpi = (dedx_range_pi.GetBinContent(bin_ - 1) +
                          dedx_range_pi.GetBinContent(bin_ + 1)) / 2

            bincmu = dedx_range_mu.GetBinContent(bin_)
            if bincmu < 1e-6:
                bincmu = (dedx_range_mu.GetBinContent(bin_ - 1) +
                          dedx_range_mu.GetBinContent(bin_ + 1)) / 2

            binepro = dedx_range_pro.GetBinError(bin_)
            if binepro < 1e-6:
                binepro = (dedx_range_pro.GetBinError(bin_ - 1) +
                           dedx_range_pro.GetBinError(bin_ + 1)) / 2

            bineka = dedx_range_ka.GetBinError(bin_)
            if bineka < 1e-6:
                bineka = (dedx_range_ka.GetBinError(bin_ - 1) +
                          dedx_range_ka.GetBinError(bin_ + 1)) / 2

            binepi = dedx_range_pi.GetBinError(bin_)
            if binepi < 1e-6:
                binepi = (dedx_range_pi.GetBinError(bin_ - 1) +
                          dedx_range_pi.GetBinError(bin_ + 1)) / 2

            binemu = dedx_range_mu.GetBinError(bin_)
            if binemu < 1e-6:
                binemu = (dedx_range_mu.GetBinError(bin_ - 1) +
                          dedx_range_mu.GetBinError(bin_ + 1)) / 2

            # errore di risoluzione su dE/dx
            errdedx = 0.04231 + 0.0001783 * trkdedx[i] * trkdedx[i]
            errdedx *= trkdedx[i]

            chi2pro += ((trkdedx[i] - bincpro) / math.sqrt(binepro**2 + errdedx**2)) ** 2
            chi2ka += ((trkdedx[i] - bincka) / math.sqrt(bineka**2 + errdedx**2)) ** 2
            chi2pi += ((trkdedx[i] - bincpi) / math.sqrt(binepi**2 + errdedx**2)) ** 2
            chi2mu += ((trkdedx[i] - bincmu) / math.sqrt(binemu**2 + errdedx**2)) ** 2

            npt += 1

    chi2s = [chi2mu / npt, chi2pro / npt, chi2ka / npt, chi2pi / npt]
    return chi2s

In [ ]:
chi2_index = 0

fig = plt.figure()

gs = fig.add_gridspec(2, 1, height_ratios=[8,2], hspace=0)

print('DATA')

filename = f'varMC_DATA{cut}.root'
file = uproot.open(filename)
tree = file['tree']
print("num entries:", tree.num_entries)
print("keys:", tree.keys())
arrays = tree.arrays(
        ["slice/_protons/_protons._dedx", 
        "slice/_protons/_protons._rr",
        "slice/_mu/_mu._dedx", 
        "slice/_mu/_mu._rr",
        "slice/_protons/_protons._theta_xw",
        "slice/_mu/_mu._theta_xw",
        "slice/_protons/_protons._depE",
        "slice/_mu/_mu._depE"],
        library="ak"
        )

chi2 = []
for i in range(len(arrays)):
    #print('slice',i)
    slice = arrays[i]

    particle_dedx = slice[f"slice/{particle}/{particle}._dedx"] #array di vettori di dedx, uno per traccia di quella particella nella slice
    particle_rr = slice[f"slice/{particle}/{particle}._rr"]

    #print('DATA: ',particle_dedx)

    if particle == '_protons' : 
        for p in range(len(particle_dedx)):
                if(len(particle_dedx[p]) > 0) :
                    chi2.append(chi2_ALG(particle_dedx[p],particle_rr[p],0,25)[chi2_index])

    if particle == '_mu' : 
        if(len(particle_dedx)) : 
            chi2.append(chi2_ALG(particle_dedx,particle_rr,0,25)[chi2_index])
    
h_data = ROOT.TH1D("h_data_chi2","",NBINS_CHI2,0,HIGH_CHI2)
h_data.Sumw2()
data_np = np.array(chi2)
for hit_data in data_np : h_data.Fill(hit_data)

h_data.Scale(1. / h_data.Integral("width"))

bin_centers_DATA = []
counts_DATA = []
errors_DATA = []

for bin in range(1,NBINS_CHI2 + 1):
    bin_centers_DATA.append(h_data.GetBinCenter(bin))
    counts_DATA.append(h_data.GetBinContent(bin))
    errors_DATA.append(h_data.GetBinError(bin))

print('OFFBEAM')

filename = f'varMC_OFFBEAM{cut}.root'
file = uproot.open(filename)
tree = file['tree']
arrays = tree.arrays(
        ["slice/_protons/_protons._dedx", 
        "slice/_protons/_protons._rr",
        "slice/_mu/_mu._dedx", 
        "slice/_mu/_mu._rr",
        "slice/_protons/_protons._theta_xw",
        "slice/_mu/_mu._theta_xw",
        "slice/_protons/_protons._depE",
        "slice/_mu/_mu._depE"],
        library="ak"
        )

chi2_OFFBEAM = []
for i in range(len(arrays)):
    slice = arrays[i]

    particle_dedx = slice[f"slice/{particle}/{particle}._dedx"] #array di vettori di dedx, uno per traccia di quella particella nella slice
    particle_rr = slice[f"slice/{particle}/{particle}._rr"]

    if particle == '_protons' : 
        for p in range(len(particle_dedx)):
                if(len(particle_dedx[p]) > 0) :
                    chi2_OFFBEAM.append(chi2_ALG(particle_dedx[p],particle_rr[p],0,25)[chi2_index])

    if particle == '_mu' : 
        if(len(particle_dedx)) : 
            chi2_OFFBEAM.append(chi2_ALG(particle_dedx,particle_rr,0,25)[chi2_index])

weights_OFFBEAM_chi2 = np.full(len(chi2_OFFBEAM), 1. / POT_OFFBEAM)

plt.subplot(gs[0])
plt.errorbar(bin_centers_DATA,counts_DATA,yerr=errors_DATA,fmt='o', ms=3.5, capsize=1.5, capthick=1.5, elinewidth=1.5, color = 'black', label = 'DATA')

m = -1
for mode in mode_vars:

    for n,sigma in enumerate(['_plus1sigma','_minus1sigma']):

        if (n == 0 and mode == 'STANDARD') or (n == 0 and mode == f'WIREMOD{cut}') : continue
        if (n == 1 and mode == 'STANDARD') or (n == 1 and mode == f'WIREMOD{cut}') : sigma = ''

        m = m+1

        filename = f'varMC_{mode}{sigma}.root'

        print(filename)

        file = uproot.open(filename)

        tree = file['tree']
        #print(tree.keys())

        arrays = tree.arrays(
        ["slice/_protons/_protons._dedx", 
        "slice/_protons/_protons._rr",
        "slice/_mu/_mu._dedx", 
        "slice/_mu/_mu._rr",
        "slice/_protons/_protons._theta_xw",
        "slice/_mu/_mu._theta_xw",
        "slice/_protons/_protons._depE",
        "slice/_mu/_mu._depE"],
        library="ak"
        )

        chi2 = []
        for i in range(len(arrays)):
            slice = arrays[i]

            particle_dedx = slice[f"slice/{particle}/{particle}._dedx"] #array di vettori di dedx, uno per traccia di quella particella nella slice
            particle_rr = slice[f"slice/{particle}/{particle}._rr"]

            #print(particle_dedx)

            if particle == '_protons' : 
                for p in range(len(particle_dedx)):
                    if(len(particle_dedx[p]) > 0) :
                        chi2.append(chi2_ALG(particle_dedx[p],particle_rr[p],0,25)[chi2_index])

            if particle == '_mu' : 
                if(len(particle_dedx)) : 
                    chi2.append(chi2_ALG(particle_dedx,particle_rr,0,25)[chi2_index])

        POT = 0
        if mode == f"WIREMOD{cut}" : POT = POT_WIREMOD 
        else : POT = POT_MC
        weights_MC_chi2 = np.full(len(chi2), 1. / POT)

        mc_off = np.concatenate([chi2, chi2_OFFBEAM])
        weights_mc_off = np.concatenate([weights_MC_chi2, weights_OFFBEAM_chi2])
        n_mc_off, bins = np.histogram(
        mc_off,
        bins=NBINS_CHI2,
        range=(0, HIGH_CHI2),
        weights=weights_mc_off,
        density=True
        )

        bin_centers = 0.5 * (bins[:-1] + bins[1:])
        bin_widths = np.diff(bins)

        ratio = np.zeros_like(counts_DATA)
        ratio_err = np.zeros_like(counts_DATA)

        for k in range(len(counts_DATA)):
            if n_mc_off[k] > 0:
                ratio[k] = counts_DATA[k] / n_mc_off[k]

        sigma_n = 0
        if sigma == '_plus1sigma' : sigma_n = 1
        elif sigma == '_minus1sigma' : sigma_n = -1

        plt.subplot(gs[0])

        lls = "-"
        llw = 2
        if mode == 'STANDARD' : 
            lls = '--'
            llw = 3
            
        plt.hist(mc_off, bins=NBINS_CHI2, range=(0, HIGH_CHI2), weights=weights_mc_off, density=True, histtype='step', lw=llw, ls=lls, label=fr'MC var {mode} ${sigma_n}\sigma$', color = var_color[m], alpha = 0.8)
        plt.setp(plt.gca().get_xticklabels(), visible=False)
        plt.gca().tick_params(labelbottom=False)
        ax = plt.gca()
        yticks = ax.get_yticks()
        # keep only ticks strictly greater than the axis minimum, so the bottom one is dropped
        ax.set_yticks(yticks[yticks > ax.get_ylim()[0]])
        plt.ylabel('entries (area norm.)', fontsize=14)
        plt.yticks(fontsize=14)

        plt.subplot(gs[1])

        dx = bin_centers[1] - bin_centers[0]

        edges = np.concatenate([
        [bin_centers[0] - dx/2],
        bin_centers[:-1] + dx/2,
        [bin_centers[-1] + dx/2]
        ])

        plt.stairs(ratio,edges,fill=False,lw=2,color=var_color[m])

        plt.fill_between(
        edges[:-1],
        1,
        ratio,
        step='post',
        alpha=0.4,
        color=var_color[m],
        lw=0
        )

        plt.axhline(1.0, color='black', linestyle='--')

        if chi2_index == 0 : which_chi2 = r'$\chi^2_{|\mu}$'
        if chi2_index == 1 : which_chi2 = r'$\chi^2_{|p}$'
        plt.xlabel(fr'{which_chi2} [MeV]', fontsize=14)
        plt.ylabel(r'$\frac{DATA}{MC}$', fontsize=14)
        plt.xticks(fontsize=14, rotation=60)
        plt.yticks(fontsize=14)
        plt.ylim(0,2)


plt.subplot(gs[0])  
plt.legend(loc='upper right',fontsize = 6, ncols = 1)
plt.savefig(f'DEDX_VAR/chi2_{chi2_index}_{particle}_WIREMOD{cut}.pdf',format='pdf',bbox_inches='tight')



        


In [ ]:
import math
import numpy as np
import uproot
import awkward as ak
import ROOT
import matplotlib.pyplot as plt

# ---------------------------------------------------------------------
# 1) Estrazione one-shot degli istogrammi PID ROOT -> array numpy
#    (va fatta UNA SOLA VOLTA, prima di qualsiasi chiamata a chi2_ALG)
# ---------------------------------------------------------------------
def hist_to_arrays(h):
    nb = h.GetNbinsX()
    content = np.array([h.GetBinContent(i) for i in range(1, nb + 1)])
    error = np.array([h.GetBinError(i) for i in range(1, nb + 1)])
    edges = np.array([h.GetBinLowEdge(i) for i in range(1, nb + 2)])  # nb+1 edges
    return content, error, edges


def fix_zero_bins(content, error):
    """Sostituisce bin con content<1e-6 con la media dei vicini (come l'originale)."""
    content = content.copy()
    error = error.copy()
    bad = np.where(content < 1e-6)[0]
    for i in bad:
        lo = max(i - 1, 0)
        hi = min(i + 1, len(content) - 1)
        content[i] = (content[lo] + content[hi]) / 2
        error[i] = (error[lo] + error[hi]) / 2
    return content, error


pro_content, pro_error, pro_edges = hist_to_arrays(dedx_range_pro)
ka_content, ka_error, ka_edges = hist_to_arrays(dedx_range_ka)
pi_content, pi_error, pi_edges = hist_to_arrays(dedx_range_pi)
mu_content, mu_error, mu_edges = hist_to_arrays(dedx_range_mu)

pro_content, pro_error = fix_zero_bins(pro_content, pro_error)
ka_content, ka_error = fix_zero_bins(ka_content, ka_error)
pi_content, pi_error = fix_zero_bins(pi_content, pi_error)
mu_content, mu_error = fix_zero_bins(mu_content, mu_error)


# ---------------------------------------------------------------------
# 2) chi2_ALG vettorizzata (nessuna chiamata ROOT dentro)
# ---------------------------------------------------------------------
def chi2_ALG(dEdx, RR, rr_min, rr_max):
    """
    Versione vettorizzata numpy di chi2_ALG.
    Ritorna [chi2mu, chi2pro, chi2ka, chi2pi] normalizzati per numero di punti usati.
    Ritorna [0,0,0,0] se non ci sono punti validi (invece di dividere per zero).
    """
    dEdx = np.asarray(dEdx, dtype=np.float64)
    RR = np.asarray(RR, dtype=np.float64)

    if len(dEdx) <= 2:
        return [0.0, 0.0, 0.0, 0.0]

    # esclude primo e ultimo punto, come nell'originale
    dEdx = dEdx[1:-1]
    RR = RR[1:-1]

    mask_rr = (RR > rr_min) & (RR < rr_max)
    trkdedx = dEdx[mask_rr]
    trkres = RR[mask_rr]

    if len(trkdedx) == 0:
        return [0.0, 0.0, 0.0, 0.0]

    valid = (trkdedx <= 100) & (trkdedx >= 0.5)
    d = trkdedx[valid]
    r = trkres[valid]

    npt = len(d)
    if npt == 0:
        return [0.0, 0.0, 0.0, 0.0]

    bin_idx = np.searchsorted(pro_edges, r, side='right') - 1
    bin_idx = np.clip(bin_idx, 0, len(pro_content) - 1)

    errdedx = (0.04231 + 0.0001783 * d * d) * d

    chi2pro = np.sum(((d - pro_content[bin_idx]) / np.sqrt(pro_error[bin_idx] ** 2 + errdedx ** 2)) ** 2)
    chi2ka = np.sum(((d - ka_content[bin_idx]) / np.sqrt(ka_error[bin_idx] ** 2 + errdedx ** 2)) ** 2)
    chi2pi = np.sum(((d - pi_content[bin_idx]) / np.sqrt(pi_error[bin_idx] ** 2 + errdedx ** 2)) ** 2)
    chi2mu = np.sum(((d - mu_content[bin_idx]) / np.sqrt(mu_error[bin_idx] ** 2 + errdedx ** 2)) ** 2)

    return [chi2mu / npt, chi2pro / npt, chi2ka / npt, chi2pi / npt]


# ---------------------------------------------------------------------
# 3) Helper: calcolo chi2 su un intero file, senza rifare slicing
#    Record awkward evento-per-evento
# ---------------------------------------------------------------------
BRANCHES = [
    "slice/_protons/_protons._dedx",
    "slice/_protons/_protons._rr",
    "slice/_mu/_mu._dedx",
    "slice/_mu/_mu._rr",
    "slice/_protons/_protons._theta_xw",
    "slice/_mu/_mu._theta_xw",
    "slice/_protons/_protons._depE",
    "slice/_mu/_mu._depE",
]


def compute_chi2_from_file(filename, particle, chi2_index):
    file = uproot.open(filename)
    tree = file['tree']
    arrays = tree.arrays(BRANCHES, library="ak")

    dedx_branch = arrays[f"slice/{particle}/{particle}._dedx"]
    rr_branch = arrays[f"slice/{particle}/{particle}._rr"]

    chi2 = []
    if particle == '_protons':
        # per ogni evento, per ogni traccia protone nell'evento
        for evt_dedx, evt_rr in zip(dedx_branch, rr_branch):
            for trk_dedx, trk_rr in zip(evt_dedx, evt_rr):
                if len(trk_dedx) > 0:
                    chi2.append(chi2_ALG(trk_dedx, trk_rr, 0, 25)[chi2_index])
    elif particle == '_mu':
        for evt_dedx, evt_rr in zip(dedx_branch, rr_branch):
            if len(evt_dedx):
                chi2.append(chi2_ALG(evt_dedx, evt_rr, 0, 25)[chi2_index])

    return chi2, tree.num_entries


# ---------------------------------------------------------------------
# 4) Script principale (stessa logica dell'originale, riorganizzata)
# ---------------------------------------------------------------------
chi2_index = 1

fig = plt.figure()
gs = fig.add_gridspec(2, 1, height_ratios=[8, 2], hspace=0)

# ---- DATA ----
print('DATA')
filename = f'varMC_DATA{cut}.root'
chi2_data, n_entries = compute_chi2_from_file(filename, particle, chi2_index)
print("num entries:", n_entries)

data_np = np.array(chi2_data)

# istogramma DATA con numpy invece di ROOT.TH1D + Fill loop
counts_DATA, bin_edges = np.histogram(data_np, bins=NBINS_CHI2, range=(0, HIGH_CHI2))
bin_centers_DATA = 0.5 * (bin_edges[:-1] + bin_edges[1:])
bin_width = bin_edges[1] - bin_edges[0]
integral = np.sum(counts_DATA) * bin_width

errors_DATA = np.sqrt(counts_DATA) / integral if integral > 0 else np.zeros_like(counts_DATA)
counts_DATA = counts_DATA / integral if integral > 0 else counts_DATA.astype(float)

# ---- OFFBEAM ----
print('OFFBEAM')
filename = f'varMC_OFFBEAM{cut}.root'
chi2_OFFBEAM, _ = compute_chi2_from_file(filename, particle, chi2_index)

weights_OFFBEAM_chi2 = np.full(len(chi2_OFFBEAM), 1. / POT_OFFBEAM)

plt.subplot(gs[0])
plt.errorbar(
    bin_centers_DATA, counts_DATA, yerr=errors_DATA,
    fmt='o', ms=3.5, capsize=1.5, capthick=1.5, elinewidth=1.5,
    color='black', label='DATA'
)

# ---- MC variations ----
m = -1
for mode in mode_vars:
    for n, sigma in enumerate(['_plus1sigma', '_minus1sigma']):

        if (n == 0 and mode == 'STANDARD') or (n == 0 and mode == f'WIREMOD{cut}'):
            continue
        if (n == 1 and mode == 'STANDARD') or (n == 1 and mode == f'WIREMOD{cut}'):
            sigma = ''

        m = m + 1

        filename = f'varMC_{mode}{sigma}.root'
        print(filename)

        chi2, _ = compute_chi2_from_file(filename, particle, chi2_index)

        POT = POT_WIREMOD if mode == f"WIREMOD{cut}" else POT_MC
        weights_MC_chi2 = np.full(len(chi2), 1. / POT)

        mc_off = np.concatenate([chi2, chi2_OFFBEAM])
        weights_mc_off = np.concatenate([weights_MC_chi2, weights_OFFBEAM_chi2])

        n_mc_off, bins = np.histogram(
            mc_off,
            bins=NBINS_CHI2,
            range=(0, HIGH_CHI2),
            weights=weights_mc_off,
            density=True
        )

        bin_centers = 0.5 * (bins[:-1] + bins[1:])

        ratio = np.zeros_like(counts_DATA)
        nonzero = n_mc_off > 0
        ratio[nonzero] = counts_DATA[nonzero] / n_mc_off[nonzero]

        sigma_n = 0
        if sigma == '_plus1sigma':
            sigma_n = 1
        elif sigma == '_minus1sigma':
            sigma_n = -1

        plt.subplot(gs[0])

        lls = "-"
        llw = 2
        if mode == 'STANDARD':
            lls = '--'
            llw = 3

        plt.hist(
            mc_off, bins=NBINS_CHI2, range=(0, HIGH_CHI2), weights=weights_mc_off,
            density=True, histtype='step', lw=llw, ls=lls,
            label=fr'MC var {mode} ${sigma_n}\sigma$', color=var_color[m], alpha=0.8
        )
        plt.setp(plt.gca().get_xticklabels(), visible=False)
        plt.gca().tick_params(labelbottom=False)
        ax = plt.gca()
        yticks = ax.get_yticks()
        ax.set_yticks(yticks[yticks > ax.get_ylim()[0]])
        plt.ylabel('entries (area norm.)', fontsize=14)
        plt.yticks(fontsize=14)

        plt.subplot(gs[1])

        dx = bin_centers[1] - bin_centers[0]
        edges = np.concatenate([
            [bin_centers[0] - dx / 2],
            bin_centers[:-1] + dx / 2,
            [bin_centers[-1] + dx / 2]
        ])

        plt.stairs(ratio, edges, fill=False, lw=2, color=var_color[m])
        plt.fill_between(
            edges[:-1], 1, ratio, step='post', alpha=0.4, color=var_color[m], lw=0
        )
        plt.axhline(1.0, color='black', linestyle='--')

        which_chi2 = r'$\chi^2_{|\mu}$' if chi2_index == 0 else r'$\chi^2_{|p}$'
        plt.xlabel(fr'{which_chi2} [MeV]', fontsize=14)
        plt.ylabel(r'$\frac{DATA}{MC}$', fontsize=14)
        plt.xticks(fontsize=14, rotation=60)
        plt.yticks(fontsize=14)
        plt.ylim(0, 2)

plt.subplot(gs[0])
plt.legend(loc='upper right', fontsize=6, ncols=1)
plt.savefig(f'DEDX_VAR/chi2_{chi2_index}_{particle}_WIREMOD{cut}.pdf', format='pdf', bbox_inches='tight')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import ROOT
from scipy.interpolate import UnivariateSpline
from scipy.interpolate import PchipInterpolator

In [ ]:
data = pd.read_csv("fitDATA.txt", sep=r"\s+")
mc = pd.read_csv("fitMC.txt", sep=r"\s+")
print(mc.columns)

#data_pro = pd.read_csv("fitDATA_pro.txt", sep=r"\s+")
#mc_pro = pd.read_csv("fitMC_pro.txt", sep=r"\s+")

In [ ]:
data_mu = pd.read_csv("fitDATA_MU.txt", sep=r"\s+")
mc_mu = pd.read_csv("fitMC_MU.txt", sep=r"\s+")

In [ ]:
mc = mc.drop(index=5).reset_index(drop=True)
data = data.drop(index=5).reset_index(drop=True)
mc_mu = mc_mu.drop(index=0).reset_index(drop=True)
data_mu = data_mu.drop(index=0).reset_index(drop=True)

In [ ]:
larghezza_data = np.sqrt(np.power(data['landau_width'],2) + np.power(data['gauss_width'],2))
larghezza_mc = np.sqrt(np.power(mc['landau_width'],2) + np.power(mc['gauss_width'],2))

e_larghezza_data = np.sqrt(
    (data['landau_width'] / larghezza_data * data['e_landau_width'])**2 +
    (data['gauss_width']  / larghezza_data * data['e_gauss_width'])**2
)

e_larghezza_mc = np.sqrt(
    (mc['landau_width'] / larghezza_mc * mc['e_landau_width'])**2 +
    (mc['gauss_width']  / larghezza_mc * mc['e_gauss_width'])**2
)

larghezza_data_mu = np.sqrt(np.power(data_mu['landau_width'],2) + np.power(data_mu['gauss_width'],2))
larghezza_mc_mu = np.sqrt(np.power(mc_mu['landau_width'],2) + np.power(mc_mu['gauss_width'],2))

e_larghezza_data_mu = np.sqrt(
    (data_mu['landau_width'] / larghezza_data_mu * data_mu['e_landau_width'])**2 +
    (data_mu['gauss_width']  / larghezza_data_mu * data_mu['e_gauss_width'])**2
)

e_larghezza_mc_mu = np.sqrt(
    (mc_mu['landau_width'] / larghezza_mc_mu * mc_mu['e_landau_width'])**2 +
    (mc_mu['gauss_width']  / larghezza_mc_mu * mc_mu['e_gauss_width'])**2
)

plt.errorbar(mc['landau_mpv'], larghezza_mc, e_larghezza_mc, fmt='o',ms=3, capsize=1.5, capthick=1.5, elinewidth=1.5, label=r'$p$ MC', color = 'cornflowerblue')
plt.errorbar(data['landau_mpv'], larghezza_data, e_larghezza_data, fmt='o',ms=3, capsize=1.5, capthick=1.5, elinewidth=1.5, label=r'$p$ DATA', color='deepskyblue')

plt.errorbar(mc_mu['landau_mpv'], larghezza_mc_mu, e_larghezza_mc_mu, fmt='o',ms=3, capsize=1.5, capthick=1.5, elinewidth=1.5, label=r'$\mu$ MC', color='darkgreen')
plt.errorbar(data_mu['landau_mpv'], larghezza_data_mu, e_larghezza_data_mu, fmt='o',ms=3, capsize=1.5, capthick=1.5, elinewidth=1.5, label=r'$\mu$ DATA', color='limegreen')

plt.xlabel('MPV [MeV]', fontsize = 18)
plt.ylabel(r'$\sqrt{(\sigma_{Landau})^2 + (\sigma_{Gauss})^2}$', fontsize = 18)
plt.xticks(fontsize=14)
plt.yticks(fontsize=14)
plt.legend(loc = 'upper left', ncols=2, fontsize=14)
plt.savefig("width_data_mc.pdf", format='pdf', bbox_inches='tight')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Parametri "veri" (li usiamo solo per generare i dati, come se fossero sconosciuti)
mu = 1
sigma1 = 1.0
sigma2 = 2.5
n_samples = 100000

# Dati già generati (è tutto ciò che "abbiamo" a disposizione)
dist1 = np.random.normal(mu, sigma1, n_samples)
dist2 = np.random.normal(mu, sigma2, n_samples)

# Stime dai dati (NON dai parametri veri)
mean2 = dist2.mean()
std2  = dist2.std()

sigma_target = sigma1  # la larghezza che vogliamo ottenere

# Riscalamento usando le stime campionarie
dist3 = mean2 + (dist2 - mean2) * (sigma_target / std2)

# Plot
plt.figure(figsize=(8, 5))
plt.hist(dist1, bins=100, density=True, histtype='step', alpha=0.6, label=f'σ={sigma1}')
plt.hist(dist2, bins=100, density=True, histtype='step', alpha=0.6, label=f'σ={sigma2}')
plt.hist(dist3, bins=100, density=True, histtype='step', alpha=0.6, label='rescaled')
plt.xlabel('x')
plt.ylabel('densità di probabilità')
plt.legend()
plt.tight_layout()
plt.show()

print(f"mean/std dist2 (stimati) = {mean2:.3f} / {std2:.3f}")
print(f"mean/std dist3 (rescaled) = {dist3.mean():.3f} / {dist3.std():.3f}  (target: {mean2:.3f} / {sigma_target})")

In [ ]:
data_mpv = ROOT.TGraphErrors()
for i in range(len(data['landau_mpv'])) :
    data_mpv.SetPoint(i,data['rr'][i],data['landau_mpv'][i])
    data_mpv.SetPointError(i,0,data['e_landau_mpv'][i])
#data_mpv_fit = ROOT.TF1("data_mpv_fit","[0]*exp(-1*[1]*x)+[2]",0,25,3)
data_mpv_fit = ROOT.TF1("data_mpv_fit","[0]*exp(x*[1]-[2]) + [3]*exp(x*[4]-[5]) + [6]",0,25,4)

fit_result = data_mpv.Fit(data_mpv_fit, "S")

x_data = np.linspace(0,25,1000)
#y = fit_result.Parameter(0) * np.exp(-1 * fit_result.Parameter(1) * x) + fit_result.Parameter(2)
y_data = fit_result.Parameter(0)*np.exp(x_data*fit_result.Parameter(1)-fit_result.Parameter(2)) + fit_result.Parameter(3)*np.exp(x_data*fit_result.Parameter(4)-fit_result.Parameter(5)) + fit_result.Parameter(6)


In [ ]:
mc_mpv = ROOT.TGraphErrors()
for i in range(len(mc['landau_mpv'])) :
    mc_mpv.SetPoint(i,mc['rr'][i],mc['landau_mpv'][i])
    mc_mpv.SetPointError(i,0,mc['e_landau_mpv'][i])
#data_mpv_fit = ROOT.TF1("data_mpv_fit","[0]*exp(-1*[1]*x)+[2]",0,25,3)
mc_mpv_fit = ROOT.TF1("mc_mpv_fit","[0]*exp(x*[1]-[2]) + [3]*exp(x*[4]-[5]) + [6]",0,25,4)

fit_result = mc_mpv.Fit(mc_mpv_fit, "S")

x_mc = np.linspace(0,25,1000)
#y = fit_result.Parameter(0) * np.exp(-1 * fit_result.Parameter(1) * x) + fit_result.Parameter(2)
y_mc = fit_result.Parameter(0)*np.exp(x_mc*fit_result.Parameter(1)-fit_result.Parameter(2)) + fit_result.Parameter(3)*np.exp(x_mc*fit_result.Parameter(4)-fit_result.Parameter(5)) + fit_result.Parameter(6)

In [ ]:
#np.sqrt(data['landau_width']*data['landau_width'] + data['gauss_width']*data['gauss_width'])

plt.errorbar(data['rr'],data['landau_mpv'],data['e_landau_mpv'],fmt='o',ms=3, capsize=1.5, capthick=1.5, elinewidth=1.5, label='DATA')
plt.errorbar(mc['rr'],mc['landau_mpv'],mc['e_landau_mpv'],fmt='o',ms=3, capsize=1.5, capthick=1.5, elinewidth=1.5, label='MC')
#plt.errorbar(data_pro['rr'],data_pro['landau_mpv'],data_pro['e_landau_mpv'],fmt='o',ms=3, capsize=1.5, capthick=1.5, elinewidth=1.5, label='DATA NEW')
#plt.errorbar(mc_pro['rr'],mc_pro['landau_mpv'],mc_pro['e_landau_mpv'],fmt='o',ms=3, capsize=1.5, capthick=1.5, elinewidth=1.5, label='MC NEW')
plt.scatter(mc['rr'],mc['expected_dedx'],s=5,label='Bethe-Block', color = 'green')
#plt.plot(x_data,y_data,label='DATA fit', color = 'pink')
#plt.plot(x_mc,y_mc,label='MC fit', color = 'violet')
plt.xlabel('Residual Range [cm]', fontsize = 18)
plt.ylabel('MPV [MeV/cm]', fontsize = 18)
plt.xticks(fontsize=14)
plt.yticks(fontsize=14)
plt.legend(loc = 'upper right', fontsize=14)
plt.savefig("mpv_pro_data_mc.pdf", format='pdf', bbox_inches='tight')


In [ ]:
plt.errorbar(data_mu['rr'],data_mu['landau_mpv'],data_mu['e_landau_mpv'],fmt='o',ms=3, capsize=1.5, capthick=1.5, elinewidth=1.5, label='DATA')
plt.errorbar(mc_mu['rr'],mc_mu['landau_mpv'],mc_mu['e_landau_mpv'],fmt='o',ms=3, capsize=1.5, capthick=1.5, elinewidth=1.5, label='MC')
plt.scatter(mc_mu['rr'],mc_mu['expected_dedx'],s=5,label='Bethe-Block', color = 'green')
#plt.plot(x,y,label='DATA fit', color = 'gray')
plt.xlabel('Residual Range [cm]', fontsize = 18)
plt.ylabel('MPV [MeV/cm]', fontsize = 18)
plt.xticks(fontsize=14)
plt.yticks(fontsize=14)
plt.legend(loc = 'upper right', fontsize=14)
plt.savefig("mpv_pro_data_mc_mu.pdf", format='pdf', bbox_inches='tight')

In [ ]:
spline_table = np.genfromtxt("spline_table.txt",dtype=[('x',float),('y',float)])
x_spline_table = spline_table['x']
y_spline_table = spline_table['y']

In [ ]:
spline_table_new = np.genfromtxt("spline_table_new.txt",dtype=[('x',float),('y',float)])
x_spline_table_new = spline_table_new['x']
y_spline_table_new = spline_table_new['y']

In [ ]:
ratio = np.asarray(mc['landau_mpv'] / data['landau_mpv'])
e_ratio = np.asarray(ratio * np.sqrt( np.power(mc['e_landau_mpv']/mc['landau_mpv'],2) + np.power(data['e_landau_mpv']/data['landau_mpv'],2) ))

ratio_mu = np.asarray(mc_mu['landau_mpv'] / data_mu['landau_mpv'])
e_ratio_mu = np.asarray(ratio_mu * np.sqrt( np.power(mc_mu['e_landau_mpv']/mc_mu['landau_mpv'],2) + np.power(data_mu['e_landau_mpv']/data_mu['landau_mpv'],2) ))

for i in range(0,len(ratio)) :
    print('[',mc['landau_mpv'][i],ratio[i],'],')
for i in range(0,len(ratio_mu)) :
    print('[',mc_mu['landau_mpv'][i],ratio_mu[i],'],')

#smoothed_ratio = []
#e_smoothed_ratio = []
#mpv_smoothed = []
#for i in range(2,len(ratio)-2) :
#    if mc['landau_mpv'][i] > 7 : continue
#    smoothed_ratio.append( (ratio[i-2] + ratio[i-1] + ratio[i] + ratio[i+1] + ratio[i+2]) / 5 )
#    e_smoothed_ratio.append( np.sqrt(np.power(e_ratio[i-2],2) +  np.power(e_ratio[i-1],2) + np.power(e_ratio[i],2) + np.power(e_ratio[i+1],2) + np.power(e_ratio[i+2],2) ) / 5 )
#    mpv_smoothed.append(mc['landau_mpv'][i])

# f(x) = A/(x-B)^2 + C, con C = 1 - A/B^2 imposto per avere f(0) = 1
#ratio_tgraph = ROOT.TGraphErrors()
#for i in range(0,len(ratio)) :
#    ratio_tgraph.SetPoint(i,mc['landau_mpv'][i],ratio[i])
#    ratio_tgraph.SetPointError(i,0,e_ratio[i])
#for i in range(0,len(ratio_mu)) :
#    ratio_tgraph.SetPoint(i,mc_mu['landau_mpv'][i],ratio_mu[i])
#    ratio_tgraph.SetPointError(i,0,e_ratio_mu[i])
#f1 = ROOT.TF1("f1", "[0]/pow(x-[1],2) + 1 - [0]/pow([1],2)", 0, 20,2)
#f1 = ROOT.TF1("f1", "[0]/pow(x-[1],2) + [2]", 3, 20)
#f1 = ROOT.TF1("f1", "[0] + [1]*log(x)", 0, 30)
#f1 = ROOT.TF1("f1", "[0] + [1]*x + [2]*x*x", 0, 20)
#f1.SetParameter(2, -4.7626e-04)
#f1.SetParameter(1, 1.2924e-02)
#f1.SetParameter(0, 9.8450e-01)


#ratio_tgraph.Fit(f1, "R")

#x_fit = np.linspace(0, 20, 1000)
#y_fit = np.array([f1.Eval(x) for x in x_fit])


#x_all = np.concatenate([mc['landau_mpv'], mc_mu['landau_mpv']])
#y_all = np.concatenate([ratio, ratio_mu])
#
#x_all = np.append(x_all,[15,15.5,16,17,18,20,22,23,24,25,26,27,28,29,30])
##y_all = np.append(y_all,[1.0742995076800181,1.0742995076800181,1.0742995076800181,1.0742995076800181,1.0742995076800181,1.0742995076800181,1.0742995076800181,1.0742995076800181,1.1,1.1,1.1,1.1])
#y_all = np.append(y_all,[1.0738063735150047,1.0738063735150047,1.0738063735150047,1.0738063735150047,1.0738063735150047,1.0738063735150047,1.0738063735150047,1.0738063735150047,1.0738063735150047,1.0738063735150047,1.0738063735150047,1.0738063735150047,1.0738063735150047,1.0738063735150047,1.0738063735150047])
#
#x_all = np.append(x_all,[0,1,1.2,1.4,1.6,1.8])
#y_all = np.append(y_all,[1,1,1,1,1,1])
#
## sort by x
#sort_idx = np.argsort(x_all)
#x_all = x_all[sort_idx]
#y_all = y_all[sort_idx]
#
#spline = UnivariateSpline(x_all, y_all, s=3)
#
#print(spline(15.5))
#
#x_spline = np.linspace(0, 15.5, 500)
#y_spline = spline(x_spline)

#final_spline = UnivariateSpline(x_spline_table, y_spline_table, s=0)
#x_final_spline = np.linspace(0, 30, 1000)
#y_final_spline = spline(x_final_spline)


plt.errorbar(mc['landau_mpv'],ratio,yerr=e_ratio,fmt='o',ms=3, capsize=1.5, capthick=1.5, elinewidth=1.5, label='protons')
#plt.plot(x_fit,y_fit,label='fit',color='gray')
#plt.plot(x_spline,y_spline,label='new spline',color='violet')
plt.plot(x_spline_table,y_spline_table,label='spline',color='green')
#plt.plot(x_spline_table_new,y_spline_table_new,label='spline new',color='violet')
#plt.plot(x_final_spline,y_final_spline,label='final spline',color='green')
#plt.errorbar(mpv_smoothed,smoothed_ratio,yerr=e_smoothed_ratio,fmt='o',ms=3, capsize=1.5, capthick=1.5, elinewidth=1.5, label='smoothed protons')
plt.errorbar(mc_mu['landau_mpv'],ratio_mu,yerr=e_ratio_mu,fmt='o',ms=3, capsize=1.5, capthick=1.5, elinewidth=1.5, label = 'muons')
plt.xlabel('MPV [MeV/cm]', fontsize = 18)
plt.ylabel('MC / DATA', fontsize = 18)
plt.xticks(fontsize=14)
plt.yticks(fontsize=14)
plt.legend(loc = 'lower right', fontsize=14)
plt.ylim(0.95,1.1)
plt.savefig("ratio_mc_data_oldspline.pdf", format='pdf', bbox_inches='tight')


In [ ]:
from scipy.optimize import brentq

def f(x):
    return spline(x) - 1

# Devi dare un intervallo [a, b] dove la funzione cambia segno
# Guardando il grafico, la curva attraversa y=1 da qualche parte
# tra x=1 e x=3 (bassi MPV) — verifica prima:
xs = np.linspace(0.5, 16, 500)
ys = spline(xs) - 1
sign_changes = np.where(np.diff(np.sign(ys)) != 0)[0]
print("cambi di segno vicino a x =", xs[sign_changes])

# poi per ogni intervallo trovato:
for i in sign_changes:
    root = brentq(f, xs[i], xs[i+1])
    print("y=1 in x =", root)

In [ ]:
x_min, x_max = 1.4127288383407872, 20.0   # l'intervallo che ti interessa
n_points = 10000            # più punti = interpolazione lineare più precisa in C++

x_export = np.linspace(x_min, x_max, n_points)
y_export = spline(x_export)

np.savetxt("spline_table.txt", np.column_stack([x_export, y_export]),
           fmt="%.8f", header="MPV MC_over_DATA", comments="")

In [ ]:
x_min, x_max = 1.2999981926531123, 15.5   # l'intervallo che ti interessa
n_points = 10000            # più punti = interpolazione lineare più precisa in C++

x_export = np.linspace(x_min, x_max, n_points)
y_export = spline(x_export)

x_add_right = np.linspace(15.51,40,1000)
y_add_right = v = np.full(1000, 1.0767303279014158)

x_add_left = np.linspace(0,1.298,1000)
y_add_left = v = np.full(1000, 1.)

x_export = np.append(x_add_left,x_export)
y_export = np.append(y_add_left,y_export)

x_export = np.append(x_export, x_add_right)
y_export = np.append(y_export, y_add_right)

np.savetxt("spline_table_new.txt", np.column_stack([x_export, y_export]),
           fmt="%.8f", header="MPV MC_over_DATA", comments="")